In [ ]:
import jax.numpy as jnp
import numpy as np
import jax
from master.symmetry import *
from master.phase_retrieval import *
from master.obj import *
from master.noise import *
from master.mle import *
import master.device as device
from master.pies import *
from master.ll_plotting import *
from master.ptychography import *
from matplotlib.patches import Circle
from matplotlib.lines import Line2D
from master.mle_pty import *
from master.operators import *
from master.spotgen import *
from master.probes import *
from master import *
from tqdm import tqdm
from master.digest import make_array_digest

from pty_defs import *

jax.config.update("jax_enable_x64", True)
cdtype = jnp.array(1j).dtype
gpu_idx, free_mb, gpu_count = device.setup_device(device_id=1)

In [ ]:
N_crop = 256
N_pad = 32
N_full = N_crop + N_pad * 2
NA_det = 0.475
NA_spot = 0.5


epsilon_PIE = 1e-12
epsilon_ML = 1e-12

frac_x = 0.80
frac_y = 0.80

L_x, L_y = 16, 16

probe_type = "regular"
probe_seed = 1776
M = 100

N_int_avg = 1e9

x_obj, obj_support = cameraman_object(N_crop, N_pad)

lambda_0 = 1.0
unit_l = 1.0

probe_params_dict = {
    "NA_spot": NA_spot,
    "frac_x": frac_x,
    "frac_y": frac_y,
    "L_x": L_x,
    "L_y": L_y,
    "probe_type": probe_type,
    "probe_seed": probe_seed,
}
probe_params = ProbeParams.from_dict(probe_params_dict)

pty_sim = PtySim(N_crop, N_pad, NA_det)
x, y, X, Y, R, k_0, k_x, k_y, K_X, K_Y, k, r_extent, k_extent = pty_sim.create_axes()
A, A_dagger, F_pupil, pupil_support = pty_sim.get_ops_and_pupil()
print(r_extent, k_extent)

yhat = A(x_obj)
plot_intensity_phase(x_obj, extent=r_extent)
plot_intensity_phase(yhat, use_log=True)
plot_intensity_phase(A_dagger(yhat) * obj_support, extent=r_extent)
plot_intensity_phase(F_pupil)

# 1. Multi-spot real-space ptychography with slight 2D scanning
Slight translations of the multi-spot pattern using a galvo-mirror should be possible. 

In [ ]:
N_full_frac = int(frac_x * N_full)
print(N_full_frac)
print(r_extent)
divisors = [i + 1 for i in range(1, N_full_frac + 1) if N_full_frac % i == 0]
divisors

In [ ]:
plot_R_probes(create_probe_Rm(64, 1.0, 0, "regular"), r=1.0)

In [ ]:
min_x, max_x, min_y, max_y = r_extent

# Define multi-spot array properties like number of spots, NA of spot and spatial area it covers
# With detector and spot NA, a bound on the total synthetic aperture NA can be given
NA_syn_opt = NA_det + NA_spot

array_bounds, array_period_x, array_period_y = calc_array_properties(
    L_x, L_y, frac_x, frac_y, r_extent
)

probes, N_photons, N_coeff = pty_sim.create_probe_with_average_intensity(
    M,
    probe_params,
    N_int_avg,
    f_probe=create_multispot_array_product_jittered,
    delta_r_coeff=0.50,
    use_shift=True,
    r_frac_jitter=0.1,
)

spectrum_filtered_probe, spectral_support, P_spectral_support = (
    pty_sim.construct_spectral_support_rel(probes[0], rel_threshold=0.01)
)

ys = A(x_obj * probes)
probed = analyse_probe(probes, x_obj, r_extent, single_plot=True)
object_probe_sum = center_crop(jnp.sum(jnp.abs(probes * x_obj) ** 2, axis=0), N_crop)
plot(object_probe_sum)
print(jnp.mean(object_probe_sum))
print(f"{N_photons:.3g}")
det_sum_unscaled = jnp.sum(jnp.abs(A(probes * x_obj)) ** 2)


obj_sum_unscaled = jnp.sum(jnp.sum(jnp.abs(probes * x_obj) ** 2, axis=0))
print(obj_sum_unscaled, det_sum_unscaled)
print(det_sum_unscaled / obj_sum_unscaled)

print(
    "Unit intensity probes and object digest: ",
    make_array_digest(probes),
    make_array_digest(x_obj),
)

In [ ]:
plot(np.abs(probes[0]))
plot(np.abs(fft2_shifted(probes[0])))

In [ ]:
plot_intensity_phase(spectrum_filtered_probe, use_log=True)
plot(spectral_support)

In [ ]:
# plot_intensity_phase(probes[3] * obj_support + 500 * obj_support)
plot_intensity_phase(probes[3] + obj_support * np.sqrt(N_int_avg))

In [ ]:
eq_per_m = int(np.sum(F_pupil))
eq_in_support = int(np.sum(spectral_support))
eq_object = 2 * eq_in_support - 1
M_min = int(np.ceil(eq_object / eq_per_m))
print(
    f"Equations per frame: {eq_per_m}\nEquations for full object support: 2 x {eq_in_support} - 1 = {eq_object}\n=> M_min = {M_min}"
)

# 2. Calculate spectral support of probe and detector together

In [ ]:
# p, N_nonzeros, spectral_support = construct_probe_inf_probe_approx(probes[0], F_pupil)
# print(N_nonzeros)
# assert N_nonzeros == 49

In [ ]:
plot(spectral_support)

In [ ]:
plot_intensity_phase(
    fft2_shifted(x_obj) * spectral_support, extent=k_extent, use_log=True
)
P_spectral_support = lambda x: ifft2_shifted(fft2_shifted(x) * spectral_support)
x_obj_est = P_spectral_support(x_obj)
plot_intensity_phase(x_obj_est, extent=r_extent)

## Run PIE on mean field ys instead of $d$

In [ ]:
beta_reg = 0.00025
alpha = 0.0125
gamma = 0.75
use_flat_init = True
x_est = pie_spectral_init(
    ys,
    probes,
    gamma=gamma,
    N_iter=400,
    epsilon=epsilon_PIE,
    alpha=alpha,
    beta_reg=beta_reg,
    spectral_support=1.0,
    obj_support=1.0,
    obj_support_init=obj_support,
    use_flat_init=use_flat_init,
    A=A,
    A_dagger=A_dagger,
)
canonicalize_est = lambda x_est: np.asarray(
    choose_representative_broadcast(P_spectral_support(x_est), x_obj_est)
)
compare_estimate(
    canonicalize_est(x_est), canonicalize_est(x_obj), N_compare_crop=N_crop
)

In [ ]:
plot_intensity_phase(fft2_shifted(x_est) * spectral_support, use_log=True)
plot_intensity_phase(fft2_shifted(x_obj_est) * spectral_support, use_log=True)

# 3. Generate measurement realizations and perform MLE refinement

In [ ]:
N_samples = 4
_, y_data = sample_from_mu(jnp.abs(ys) ** 2, batch_dim=N_samples)

In [ ]:
# Was real before, causing reconstruction bugs!
x_pies = jnp.zeros((N_samples,) + x_est.shape, cdtype)
for i in tqdm(range(N_samples)):
    y_complex = jnp.astype(y_data[i, :], cdtype)
    # Already choose the best representative
    x_pies = x_pies.at[i, :].set(
        pie_spectral_init(
            y_complex,
            probes,
            gamma=gamma,
            N_iter=500,
            epsilon=epsilon_PIE,
            spectral_support=1.0,
            obj_support=1.0,
            obj_support_init=obj_support,
            use_flat_init=use_flat_init,
            alpha=alpha,
            beta_reg=beta_reg,
            A=A,
            A_dagger=A_dagger,
        )
    )
    if i == 0:
        print(y_complex.shape, x_obj.shape)
        print(y_data.dtype, probes.dtype, y_complex.dtype, x_pies.dtype)
        plot_intensity_phase(x_pies[i, :])

In [ ]:
print(x_pies.shape, probes.shape, y_data.shape, obj_support.shape)
x_model_ml_second_order, lls_second_order = mle_ptycho_second_order_consensus(
    x_pies,
    probes,
    y_data,
    obj_support=1.0,
    alpha=1.0,
    N_iter=1000,
    epsilon=epsilon_ML,
    track_ll=True,
    A=A,
    A_dagger=A_dagger,
)
x_model_ml_second_order = canonicalize_est(x_model_ml_second_order)

In [ ]:
plot_ll(lls_second_order)

In [ ]:
x_gt = jnp.broadcast_to(
    x_obj,
    (N_samples,) + x_obj_est.shape,
)
x_model_ml_second_order_gt, lls_second_order_gt = mle_ptycho_second_order_consensus(
    x_gt,
    probes,
    y_data,
    obj_support=1.0,
    alpha=1.0,
    N_iter=1000,
    epsilon=epsilon_ML,
    track_ll=True,
    A=A,
    A_dagger=A_dagger,
)
x_model_ml_second_order_gt = P_spectral_support(x_model_ml_second_order_gt)

In [ ]:
plot_ll(lls_second_order_gt)

In [ ]:
lim = lambda x: center_crop(x, N_crop)
plot_intensity_phase(fft2_shifted(x_model_ml_second_order[-1]), use_log=True)
plot_intensity_phase(fft2_shifted(x_obj_est), use_log=True)
plot_intensity_phase(x_model_ml_second_order[-1], use_log=False)
plot_intensity_phase(lim(x_model_ml_second_order[-1]), use_log=False)

# Estimator statistics
First evaluates plain PIE output, then PIE followed by ML stage.

In [ ]:
def evaluate_estimator(x_model_ml, x_obj, obj_support, label="HIO+ML", extent=None):
    n_hat = np.abs(x_model_ml) ** 2 - jnp.var(x_model_ml, axis=0, ddof=1, keepdims=True)
    n = jnp.abs(x_obj) ** 2
    n_hat_mu, n_hat_std, n_hat_bias, n_hat_MSE = estimator_stats(n_hat, n)
    print("n_k")
    plot(n_hat_mu, label=r"$[\mu_{\hat{n}_k}]^{" + label + r"}$", extent=extent)
    plot(n, label=r"$n_k$", extent=extent)
    plot(n_hat_bias, label=r"$[\mu_{\hat{n}_k}]^{" + label + r"} - n_k$", extent=extent)
    plot(n_hat_std, label=r"$[\sigma_{\hat{n}_k}]^{" + label + r"}$", extent=extent)
    plot(n_hat_MSE, label=r"$[MSE_{\hat{n}_k}]^{" + label + r"}$", extent=extent)

    print("phi_k")

    phi_hat = np.angle(x_model_ml) * obj_support
    phi = jnp.angle(x_obj) * obj_support

    # Circular mean of the phase estimates
    phi_hat_mu = np.angle(np.mean(np.exp(1j * phi_hat), axis=0))

    # Wrapped phase errors in (-pi, pi]
    phi_hat_error = np.angle(np.exp(1j * (phi_hat - phi)))

    # Statistics in the local phase coordinate
    phi_hat_bias = np.mean(phi_hat_error, axis=0)
    phi_hat_std = np.std(phi_hat_error, axis=0, ddof=1)
    phi_hat_MSE = np.mean(phi_hat_error**2, axis=0)

    plot(phi_hat_mu, label=r"$[\mu_{\hat{\phi}_k}]^{" + label + r"}$", extent=extent)
    plot(phi, label=r"$\phi_k$", extent=extent)
    plot(
        phi_hat_bias,
        label=r"$[\mu_{\hat{\phi}_k}]^{" + label + r"} - \phi_k$",
        extent=extent,
    )
    plot(
        phi_hat_std, label=r"$[\sigma_{\hat{\phi}_k}]^{" + label + r"}$", extent=extent
    )
    plot(phi_hat_MSE, label=r"$[MSE_{\hat{\phi}_k}]^{" + label + r"}$", extent=extent)
    return n_hat_mu, n_hat_std, phi_hat_mu, phi_hat_std

In [ ]:
evaluate_estimator(
    lim(canonicalize_est(x_pies)), lim(x_obj_est), lim(obj_support), label="rPIE"
)

In [ ]:
evaluate_estimator(
    lim(x_model_ml_second_order), lim(x_obj_est), lim(obj_support), label="rPIE+FS"
)

In [ ]:
evaluate_estimator(
    fft2_shifted(x_model_ml_second_order), fft2_shifted(x_obj_est), 1.0, label="rPIE+FS"
)

In [ ]:
evaluate_estimator(
    lim(x_model_ml_second_order_gt), lim(x_obj_est), lim(obj_support), label="ORACLE+FS"
)

In [ ]:
(
    u_hat_mu,
    u_hat_std,
    u_hat_bias,
    u_hat_MSE,
    v_hat_mu,
    v_hat_std,
    v_hat_bias,
    v_hat_MSE,
) = evaluate_estimator_uv(
    center_crop(x_model_ml_second_order_gt, N_crop),
    center_crop(x_obj_est, N_crop),
    center_crop(obj_support, N_crop),
    1.0,
    label="ORACLE+ML",
    extent=r_extent,
)